In [1]:
"""
knobs left to right
74 A
71 D
76 S
77 R
93 env depth
18 env invert
19 env select
16 LFO select
17

sliders left to right
73 sub
75 co
79 res
72 pwm
80 external
81 LFO rate
82 LFO depth
83 LFO shape
85

buttons (0 or 127 only)
24
25
26
27
20
21
22
23
"""

# categories: VOICE, LFO1-5, ADSR1-5

# virtual patch matrix inside voice: pitch:LFO1, co:ADSR2, res:LFO3
# stretch goal: co:ADSR2 * LFO1  combine(ADSR2, LFO1)

# variable names inside voice objects:
"""
        self.xfade = 0
        self.suboctave = 0
        self.pwm = 0
        self.envelope = 0
        self.resonance = 0
        self.cutoff = 0
"""



"""
# in voice class:
self.__setattr__(q) = CONTROLS_DICT[q]
"""

adsr_parameter_mapping = {
    74: "a",
    71: "d",
    76: "s",
    77: "r",
    93: "depth"
}

lfo_parameter_mapping = {81: "rate", 82: "depth", 83: "shape"}

voice_parameter_mapping = {
    73: "sub",
    75: "co",
    79: "res",
    72: "pwm",
    80: "external"
}

option_lists = {"shape":["SAW", "RAMP", "TRI", "SINE", "SHARK"],
                "invert": ["ON", "OFF"]
               }

adsr_keys = list(adsr_parameter_mapping.keys())
lfo_keys = list(lfo_parameter_mapping.keys())
voice_keys = list(voice_parameter_mapping.keys())

objtypes = {}  # maps a numeric channel to a tuple of name and a dict of all its possible parameters
# lets us look up what kind of object we are controlling based on a hardcoded control number
for k in adsr_keys:
    objtypes[k] = ("ADSR", adsr_parameter_mapping)
for k in lfo_keys:
    objtypes[k] = ("LFO", lfo_parameter_mapping)
for k in voice_keys:
    objtypes[k] = ("VOICE", voice_parameter_mapping)

def listindex(ls, val):

    """use a value of 0-255 to look up something in a list. Regardless of list length, 255 is divided evenly among its items"""

    index = (len(ls) * val) // 255
    return ls[index]

class Controls:

    def __init__(self, voice_list, lfo_list, adsr_list):

        """Pass in lists of modulators which were instantiated in the main thread. We will just refer to these by number index 1..5"""

        self.voice_list = voice_list
        self.lfo_list = lfo_list
        self.adsr_list = adsr_list
        self.selected_property = None  # what variable are we currently looking at?
        self.selected_adsr = adsr_list[0]  # indexing the list we were passed on instantiation
        self.selected_lfo = lfo_list[0]
        self.selected_voice = voice_list[0]
        self.selected_object = ""  # this is just a text label to show on the LCD what we are editing
        self.updated = []  # a queue of updated parameters to provide to the main loop when asked
        # tuples of object, property and value
        # mainloop will do object.__setattr__(value)
        self.printable_names = self.make_printable_names()

    def make_printable_names(self):

        dc = {}
        for i, j in enumerate(self.voice_list):
            dc[j] = f"VOICE {i+1}"
        for i, j in enumerate(self.lfo_list):
            dc[j] = f"LFO {i+1}"
        for i, j in enumerate(self.adsr_list):
            dc[j] = f"ADSR {i+1}"
        return dc
        
    def process_control_signal(self, channel, value):

        # these won't be set if we are changing the selected ADSR/LFO, still need to update
        # the display with the identity of the changed object though
        param_name = None
         
        if channel == 17:  #  param select knob, 
            pass
        elif channel == 85:  #generic entry for any value
            pass
        elif channel == 19: # envelope select
            self.selected_adsr = listindex(self.adsr_list, value)
            self.selected_object = "ADSR"
            actual_object = self.selected_adsr
            value = None
        elif channel == 16:  # lfo select
            self.selected_lfo = listindex(self.lfo_list, value)
            self.selected_object = "LFO"
            actual_object = self.selected_lfo
            value = None
        else:
            object_type, object_params = objtypes[channel]            
            param_name = object_params[channel]
    
            self.selected_property = param_name
            self.selected_object = object_type
    
            if object_type == "VOICE":
                actual_object = self.selected_voice
            elif object_type == "LFO":
                actual_object = self.selected_lfo
            elif object_type == "ADSR":
                actual_object = self.selected_adsr

        self.updated.append((actual_object, param_name, value))  # this is used to update objects in the main thread

    def get_updated(self):

        """Returns a list of variables that have changed since the last time we checked. The main loop uses this to 
        tell objects they need to update themselves"""

        out = []
        while len(self.updated) > 0:
            out.append(self.updated.pop())
        return out
        


In [2]:
class DummyDisplay:

    def __init__(self):

        self.line1 = [" "] * 16
        self.line2 = [" "] * 16

    def printout(self):

        return f"{"".join(self.line1)}\n{"".join(self.line2)}"

    def update(self, diff1, diff2):

        self.update_line(self.line1, diff1)
        self.update_line(self.line2, diff2)

    def update_line(self, line, diff):

        for start, run in diff:
            ind = start
            for letter in run:
                #print(f"Writing{letter} at index {ind}")
                line[ind] = letter
                ind += 1

In [3]:
class DisplayManager:

    def __init__(self, voice_list, lfo_list, adsr_list):

        self.voice_list = voice_list
        self.lfo_list = lfo_list
        self.adsr_list = adsr_list
        self.line1 = ""
        self.line2 = ""

    def update(self, update_tup):

        line1, line2 = self.get_lines(update_tup)
        diff1 = self.diff_line(self.line1, line1)
        diff2 = self.diff_line(self.line2, line2)

        self.line1 = line1
        self.line2 = line2

        return diff1, diff2  # lists of tuples [(index, run of characters)]
        # this lets us only update the LCD characters that have changed

    def get_lines(self, update_tup):

        actual_object, param_name, value = update_tup  # unpacking the tuple generated by the Controls object
        printable_index = 999  # a placeholder indicator that something has gone very wrong
        for ls in self.voice_list, self.lfo_list, self.adsr_list:
            if actual_object in ls:
                # so the user can see LFO1, LFO2...
                printable_index = ls.index(actual_object) +1
                break

        if actual_object in self.adsr_list or actual_object in self.lfo_list:
            # these objects are printable so we just pretty print them and insert their number
            outa, outb = actual_object.pretty_print()
            outa %= printable_index

        elif actual_object in self.voice_list:
            # can't fit all voice params on one display so we just display one at a time
            outa = "VOICE %i" % printable_index
            outb = "%s: %i" % (param_name, value)  # todo: pretty conversion e.g. cutoff into Hz

        return outa, outb

    def diff_line(self, old_line, new_line):

        """Detect only the characters that changed"""
        
        while len(new_line) < len(old_line):
            new_line += " "  # add spaces to overwrite the longer old line

        oldlen = len(old_line)
        newlen = len(new_line)
        
        minlen = min(oldlen, newlen)

        runs = []  # start index and a run of characters that need to be replaced
        index = 0

        runstart = 0
        run = []
        accumulating = False
        
        while index < minlen:
            old = old_line[index]
            new = new_line[index]
            
            if old == new:
                if run:  # don't append the empty list first time round
                    accumulating = False
                    runs.append((runstart, run))
                    runstart = 0
                    run = []
            else:
                if not accumulating:
                    accumulating = True
                    runstart = index
                run.append(new)
                
            index += 1

        if accumulating:  # make sure we catch the run if the line was different right
            # up until the last character
            runs.append((runstart, run))

        if oldlen < len(new_line):
            runs.append((oldlen, new_line[oldlen:]))
            
        return runs

In [4]:
import ADSR2, LFO2
from importlib import reload
reload(ADSR2)
reload(LFO2)

adsrlist = [ADSR2.ADSR(), ADSR2.ADSR(),ADSR2.ADSR(),ADSR2.ADSR(),ADSR2.ADSR()]
lfolist = [LFO2.LFO(), LFO2.LFO(), LFO2.LFO(), LFO2.LFO()]
voicelist = [LFO2.LFO()]

C = Controls(voicelist, lfolist, adsrlist)

def send_signal(chan, val):

    C.process_control_signal(chan, val)
    return C.get_updated()

events = [(81, 122),
(82, 94),
(16, 200),
(81, 132),
(83, 254),
(82, 12),
(19, 12),
(93, 64),
(74, 122),
(81, 172),
(73, 172),
(72, 255),
(81, 150)]

DM = DisplayManager(voicelist, lfolist, adsrlist)
DD = DummyDisplay()

for e in events:
    C.process_control_signal(*e)
    ret = C.get_updated()
    print(ret)
    ob, parm, value = ret[0]
    if parm:
        ob.__setattr__(parm, value)
    outa, outb = DM.update(ret[0])
    print(outa, outb)
    DD.update(outa, outb)
    print(DD.printout())
    #print(ret[0][0].pretty_print())
    print("=====")




    

[(<LFO2.LFO object at 0x000001E8B0F2C980>, 'rate', 122)]
[(0, 'LFO 1 (SAW)')] [(0, 'R:122 D:100%')]
LFO 1 (SAW)     
R:122 D:100%    
=====
[(<LFO2.LFO object at 0x000001E8B0F2C980>, 'depth', 94)]
[] [(8, ['3', '6', '%', ' '])]
LFO 1 (SAW)     
R:122 D:36%     
=====
[(<LFO2.LFO object at 0x000001E8B0EB6780>, None, None)]
[(4, ['4'])] [(4, ['8']), (8, ['1', '0', '0']), (11, '%')]
LFO 4 (SAW)     
R:128 D:100%    
=====
[(<LFO2.LFO object at 0x000001E8B0EB6780>, 'rate', 132)]
[] [(3, ['3', '2'])]
LFO 4 (SAW)     
R:132 D:100%    
=====
[(<LFO2.LFO object at 0x000001E8B0EB6780>, 'shape', 254)]
[(8, ['H', 'A', 'R']), (11, 'KFIN)')] []
LFO 4 (SHARKFIN)
R:132 D:100%    
=====
[(<LFO2.LFO object at 0x000001E8B0EB6780>, 'depth', 12)]
[] [(8, ['4', '%', ' ', ' '])]
LFO 4 (SHARKFIN)
R:132 D:4%      
=====
[(<ADSR2.ADSR object at 0x000001E8B0F2C6E0>, None, None)]
[(0, ['A', 'D', 'S', 'R', ' ', '1', ' ', '(', '1', '0', '0', '%', ')', ' ', ' ', ' '])] [(0, ['1', '2', '7', ' ', '1', '2', '7', ' ', 

In [14]:
def build_instruction_queue(ls, line):

    """given a list of update tuples, break it into instructions to be sent to the LCD screen"""
    # only need to position the cursor once, then it auto-increments
    # need to tell it whether it's on line 0 (top) or 1 to set the y address appropriately

    out = []
    for loc, chrs in ls:
        out.append((0x80 + 0x40 * y + loc, 0))  # position cursor and 0 = command, not data
        for c in chrs:
            out.append((c, 1))
    
            

In [10]:
DTEST = DisplayManager(voicelist, lfolist, adsrlist)
DTEST.diff_line("        ", "abc3555555555555555555555555")

[(0, ['a', 'b', 'c', '3', '5', '5', '5', '5']), (8, '55555555555555555555')]

In [96]:
"abcdefghabcdefgh"[15]

'h'

In [62]:
mys

'abc6de'

In [63]:
"av%sasdadw" % "goon"

'avgoonasdadw'

In [5]:
65536 >> 8

256

In [7]:
((65536 - 32768) / 65535.0 * 1E7) / 200

25000.381475547416

In [9]:
def cv2freq(cv):

    return 10**(7.27E-3 * cv - 0.64)

cv2freq(255)

16.362512826483755

In [10]:
cv2freq(0)

0.2290867652767773

In [20]:
import math

def freq2cv(freq):

    """Expects frequency in Hz"""

    return (math.log10(freq/1000.0) + 0.64) / 7.27E-3

In [22]:
freq2cv(16000)

253.66162072296075

In [23]:
freq2cv(231)

0.49683354775025385

In [27]:
import filtertable

filtertable.FILTER_CVS

array('B', [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 4, 7, 11, 14, 18, 21, 25, 28, 32, 35, 38, 42, 45, 49, 52, 56, 59, 63, 66, 70, 73, 76, 80, 83, 87, 90, 94, 97, 101, 104, 108, 111, 114, 118, 121, 125, 128, 132, 135, 139, 142, 145, 149, 152, 156, 159, 163, 166, 170, 173, 177, 180, 183, 187, 190, 194, 197, 201, 204, 208, 211, 214, 218, 221, 225, 228, 232, 235, 239, 242, 246, 249, 252])